# Map Pipeline — Durable Resume

Resumes the interrupted first canonical build at the first component without
a durable `pipeline.map_pipeline.component_complete` marker. Completed
foundation, clinical, event and early maternity components are never rerun.

In [0]:
%pip install openai pyarrow

In [0]:
%restart_python

In [0]:
for _name, _default in {
    "pipeline_run_id": "",
    "force_full_refresh": "true",
    "create_cutover_backups": "true",
    "run_post_deployment_checks": "true",
}.items():
    try:
        dbutils.widgets.text(_name, _default)
    except Exception:
        pass

In [0]:
%run "/Workspace/Shared/ADC-DB/Prod/Pipelines/Bronze/_map_common"

In [0]:
_RESUME_REQUIRED_COMPLETE = {
    "map_address": [
        "4_prod.bronze.map_address__canonical",
        "4_prod.bronze.map_address_epc__canonical",
    ],
    "map_person": ["4_prod.bronze.map_person__canonical"],
    "map_care_site": ["4_prod.bronze.map_care_site__canonical"],
    "map_medical_personnel": [
        "4_prod.bronze.map_medical_personnel__canonical"
    ],
    "map_encounter": ["4_prod.bronze.map_encounter__canonical"],
    "map_diagnosis": ["4_prod.bronze.map_diagnosis__canonical"],
    "map_problem": ["4_prod.bronze.map_problem__canonical"],
    "map_med_admin": ["4_prod.bronze.map_med_admin__canonical"],
    "map_procedure": ["4_prod.bronze.map_procedure__canonical"],
    "map_death": ["4_prod.bronze.map_death__canonical"],
    "map_numeric_events": ["4_prod.bronze.map_numeric_events__canonical"],
    "map_date_events": ["4_prod.bronze.map_date_events__canonical"],
    "map_text_events": ["4_prod.bronze.map_text_events__canonical"],
    "map_nomen_events": ["4_prod.bronze.map_nomen_events__canonical"],
    "map_coded_events": ["4_prod.bronze.map_coded_events__canonical"],
    "map_mat_pregnancy": [
        "4_prod.bronze.map_mat_pregnancy__canonical"
    ],
    "map_mat_birth": ["4_prod.bronze.map_mat_birth__canonical"],
    "map_mat_vte_assessment": [
        "4_prod.bronze.map_mat_vte_assessment__canonical"
    ],
    "map_family_history": [
        "4_prod.bronze.map_family_history__canonical"
    ],
}

_resume_missing = []
_resume_completed = []
for _component, _targets in _RESUME_REQUIRED_COMPLETE.items():
    if _pipeline_resume_component_complete(_component, _targets):
        _resume_completed.append(_component)
    else:
        _resume_missing.append({"component": _component, "targets": _targets})

if _resume_missing:
    raise RuntimeError(
        "Resume boundary is unsafe: one or more pre-patient-journey components "
        "lack durable completion markers. Refusing to skip them: "
        + _pipeline_json.dumps(_resume_missing, default=str)
    )

print(
    "[RESUME] Verified durable completion markers for "
    f"{len(_resume_completed)} prior components. Starting Map 40 at the first "
    "incomplete component."
)
_pipeline_audit(
    None,
    "RUN_RESUME_START",
    {
        "completed_components_skipped": _resume_completed,
        "resume_notebook": "New Bronze/map_pipeline",
    },
)

In [0]:
%run ./map_40_maternity_journey

In [0]:
%run ./map_50_pathology

In [0]:
_pipeline_compatibility_metrics = _pipeline_publish_all_compatibility_targets()
_pipeline_commit_all_checkpoints()
_pipeline_audit(
    None,
    "RUN_SUCCESS",
    {
        "resume": True,
        "completed_components_skipped": _resume_completed,
        "updated_targets": sorted(set(_PIPELINE_UPDATED_TARGETS)),
        "compatibility_metrics": _pipeline_compatibility_metrics,
        "cutover_backups": _PIPELINE_CUTOVER_BACKUPS,
    },
)

_map_pipeline_summary = {
    "status": "SUCCESS",
    "pipeline": "map_pipeline_resume",
    "run_id": _PIPELINE_RUN_ID,
    "full_refresh": bool(_PIPELINE_FULL_REFRESH),
    "completed_components_skipped": _resume_completed,
    "updated_targets": sorted(set(_PIPELINE_UPDATED_TARGETS)),
    "compatibility_metrics": _pipeline_compatibility_metrics,
    "cutover_backups": _PIPELINE_CUTOVER_BACKUPS,
}
print(_pipeline_json.dumps(_map_pipeline_summary, default=str))
dbutils.notebook.exit(
    _pipeline_json.dumps(_map_pipeline_summary, default=str)
)